# 02 · Extract e carga na camada raw

1. **Extract**: baixar os 9 CSVs do Kaggle (ou reaproveitar os que já estão em `data/raw/`).
2. **Load raw**: copiar cada CSV, quase sem alterações, para o schema `raw` do Postgres.

schema `raw` é a prova d origem (não corrige)

In [1]:
# Configuracao inicial
import os
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")


def find_project_root():
    current = Path.cwd().resolve()
    for p in [current] + list(current.parents):
        if (p / "src").exists() and (p / "data").exists() and (p / "config").exists():
            return p
    raise RuntimeError("Raiz do projeto nao encontrada")


def project_path(*segments):
    return find_project_root().joinpath(*segments)


root = find_project_root()
os.chdir(root)
print(f"Diretorio de trabalho: {root}")


Diretorio de trabalho: C:\Users\user\Downloads\Códigos\olist-ecommerce-pipeline\Template


## 1. Extract

Precisa de `KAGGLE_USERNAME`/`KAGGLE_KEY` no `.env` (veja `.env.example`). Se os CSVs já estiverem em
`data/raw/`, o download é pulado e então rodar esta célula de novo não tem custo

In [2]:
from src.etl.kaggle_ingestion import KaggleIngestion

ingestion = KaggleIngestion()
files = ingestion.download_dataset(dest_dir=str(project_path("data", "raw")))
files

['olist_customers_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'product_category_name_translation.csv']

## 2. Load raw

`RawLoader` lê cada CSV e cria uma tabela em `raw.<nome>` com as mesmas colunas e tipos que o
`pandas.read_csv` escolheu

In [3]:
from src.etl.db import get_engine
from src.etl.raw_loader import RawLoader

engine = get_engine()
loader = RawLoader(engine, source_dir=str(project_path("data", "raw")))
row_counts = loader.load_all()
row_counts

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 3. Conferindo no banco

In [4]:
tables = pd.read_sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'raw' ORDER BY 1", engine)
print(f"{len(tables)} tabelas em raw:", ", ".join(tables["table_name"]))

pd.read_sql("SELECT customer_zip_code_prefix FROM raw.customers LIMIT 5", engine).assign(
    tipo_no_pandas=lambda d: str(d["customer_zip_code_prefix"].dtype)
)

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)